# vKF and KF from the joint posterior, Laplacian noise

Section 3 of the new `main.tex` derives four filters from the **joint** posterior, reduced to the
scalar $s_t = \bx_t^{\mathsf T}(\btheta_t - \bw_{t-1})$. They differ only in the shape of the
predicted covariance $\tilde\bSigma_t$; the noise enters through the same two scalars,
$\chi_t(e_t)$ and $\chi_t'(e_t)$.

| family | $\tilde\bSigma_t$ | cost | status |
|---|---|---|---|
| KF | full | $O(M^2)$ | **issue #11, here** |
| vKF | diagonal | $O(M)$ | **issue #10, here** |
| sKF | $\tilde v_t\bI$ | $O(M)$ | done, Ignacio's notebook 07 |
| fKF | $(\bar v + \varepsilon)\bI$ | $O(M)$ | the fixed family of the convergence analysis |

The KF is the filter the marginal route structurally **could not** produce, since that route
required a factorized prior, and it is the reason for the pivot. Section 4 claims it recovers the
loss under coloured input **without prewhitening**, because the update direction
$\tilde\bSigma_t\bx_t$ is decorrelated and $\sigma_t^2$ is then correct in every direction. Section 5
below tests that directly on the AR($-0.9$) input of notebook 03.

**Convention.** $b_\eta = \sqrt{v_\eta/2}$, the old one, so that these runs sit beside notebooks 03
and 07 unchanged. Issue #9 switches to $b_\eta = \mathrm{E}|\eta_t|$ and re-runs; `b_eta` is a
parameter everywhere below and is never hardcoded.

## 1. Imports

In [ ]:
import time

import numpy as np
from matplotlib import pyplot as plt
from scipy.signal import lfilter
from scipy.special import gammaln, log_ndtr
from scipy.stats import gennorm
import rir_generator as rir

%config InlineBackend.figure_format = 'svg'
COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']
NOTEBOOK_START = time.time()
print("imports ready")

## 2. The scenario

Notebook 03 unchanged: $M = 128$ room impulse response of section 7.1, AR($-0.9$) input of unit
variance, generalized Gaussian noise at $\beta^* = 0.2$ and SNR 5 dB, $N = 96000$, $R = 20$, target
misalignment $-20$ dB. The input correlation is a knob, so that section 5 can run the same scenario
with white input.

In [ ]:
M = 128                     # filter length / length of the impulse response
N = 96000                   # steps per run, as in notebook 03
R = 20                      # independent realisations for the checked runs
R_SEARCH = 3                # realisations while scanning the epsilon grid
N_SEARCH = 24000            # scan length
WARMUP = 500                # AR samples discarded so the input starts stationary
AR_A = -0.9                 # AR(1) coefficient of the input
SNR_DB = 5.0                # as in Fig. 3 of the paper
BETA = 0.2                  # shape of the generalized Gaussian measurement noise
VAR_THETA_0 = 2.0           # initial prior variance on each weight
TARGET_DB = -20.0           # target misalignment
FS = 8000
ROOM, T60, C_SOUND, SRC, MIC = [5, 10, 6], 0.2, 340, [1, 2.5, 2], [1, 1.5, 1]

ho = rir.generate(c=C_SOUND, fs=FS, r=MIC, s=SRC, L=ROOM,
                  reverberation_time=T60, nsample=M).flatten()
ho = ho/np.linalg.norm(ho)


def correlation(ar_a, m=M):
    lags = np.abs(np.subtract.outer(np.arange(m), np.arange(m)))
    return ar_a**lags if ar_a != 0 else np.eye(m)


def b_eta_of(var_eta, convention):
    """The Laplacian scale the filter is told, under either convention (issue #9).

    "variance": b = sqrt(v_eta/2), matching the noise variance. What every notebook up to 07 used.
    "mad":      b = E|eta_t|, matching the mean absolute deviation. The maximum-likelihood and
                KL-closest Laplacian fit to the actual noise, and what the new draft uses."""
    if convention == "variance":
        return np.sqrt(var_eta/2)
    scale = np.sqrt(var_eta/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))
    return scale*np.exp(gammaln(2/BETA) - gammaln(1/BETA))


def scenario(ar_a, convention="variance"):
    """Signal power, noise variance and Laplacian scale at this input correlation."""
    P_signal = float(ho @ correlation(ar_a) @ ho)
    var_eta = P_signal/10**(SNR_DB/10)
    return P_signal, var_eta, b_eta_of(var_eta, convention)


def generate_signals(ar_a, seed, n=N):
    """generate_signals of notebook 03, with the AR coefficient as a knob (0 = white).

    The noise depends on the SNR alone, not on which b_eta the filter is told."""
    _, var_eta, _ = scenario(ar_a)
    rng = np.random.default_rng(seed)
    u = np.sqrt(1 - ar_a**2)*rng.standard_normal(N + WARMUP)
    x = lfilter([1.0], [1.0, -ar_a], u)[WARMUP:]
    scale = np.sqrt(var_eta/np.exp(gammaln(3/BETA) - gammaln(1/BETA)))
    d = np.convolve(ho, x)[:N] + gennorm.rvs(BETA, scale=scale, size=N, random_state=rng)
    return x[:n], d[:n]


for ar_a in (AR_A, 0.0):
    P, v, b = scenario(ar_a)
    ev = np.linalg.eigvalsh(correlation(ar_a))
    print(f"AR a = {ar_a:>4}:  P_signal = {P:.4f}  var_eta = {v:.4e}  b_eta = {b:.4e}"
          f"  cond(Rxx) = {ev.max()/ev.min():.1f}")

## 3. The two scalars

Copied read-only from Ignacio's notebook 07: the whole of the noise model lives in these two
functions, and all four families call them with their own $\sigma_t$.

In [ ]:
# === IGNACIO: log_mills / chi_laplacian - copied verbatim, NOT edited ===
# source: branch ignacio/joint-vs-marginal-vs-minorized,
#         notebooks/07_skf_conjunto.ipynb, commit 9406077
def log_mills(z):
    """log R(z), with R(z) = Phi(-z)/phi(z) the Mills ratio, eq. (40). R grows like e^{z^2/2} for
    negative z and overflows, so it is only ever handled through its logarithm."""
    return log_ndtr(-z) + 0.5*z**2 + 0.5*np.log(2*np.pi)


def chi_laplacian(e, sigma, b_eta):
    """Correction chi_t(e_t) and its slope chi'_t(e_t) for Laplacian noise, eqs. (39), (41) and (43)."""
    u = e/sigma                                        # u_t = e_t / sigma_t
    k_t = sigma/b_eta                                  # k_t = sigma_t / b_eta
    tau = sigma**2/b_eta                               # tau_t = sigma_t^2 / b_eta, the largest correction
    log_R_minus = log_mills(k_t - u)                   # log R(k_t - u_t)
    log_R_plus = log_mills(k_t + u)                    # log R(k_t + u_t)
    Lambda = np.tanh((log_R_minus - log_R_plus)/2)     # (R- - R+)/(R- + R+), in (-1, 1)
    chi = tau*Lambda                                   # chi = tau Lambda
    chi_slope = (2*k_t*np.exp(-np.logaddexp(log_R_minus, log_R_plus))   # 2k / (R- + R+)
                 - k_t**2*(1 - Lambda**2))                               # - k^2 (1 - Lambda^2)
    return chi, chi_slope
# === end of the copied block ===


def chi_gaussian(e, sigma, v_eta):
    """Eq. (44), the Gaussian instance. Used only for the Kalman cross-check of section 4."""
    frac = sigma**2/(sigma**2 + v_eta)
    return frac*e, frac


print("chi ready")

## 4. The filters

Two forms of each. The **scalar** form carries the shared signature
`filter(n, x, d, w0, parameters) -> {"w_hist", "e"}`, so `run_filter` and `sweep_filter` of notebook
07 take it unchanged. The **batched** form runs $B$ filters side by side, one per grid value and
realisation, and is used for the sweeps only; section 4 checks the two agree.

### vKF, the diagonal family, eqs. (37)-(38)

$$\tilde v_{t,m} = v_{t-1,m} + \varepsilon, \qquad
  \sigma_t^2 = \sum_m \tilde v_{t,m}x_{t,m}^2, \qquad
  w_{t,m} = w_{t-1,m} + \frac{\tilde v_{t,m}x_{t,m}}{\sigma_t^2}\chi_t(e_t), \qquad
  v_{t,m} = \tilde v_{t,m}\Big(1 - \chi_t'(e_t)\frac{\tilde v_{t,m}x_{t,m}^2}{\sigma_t^2}\Big)$$

### KF, the full covariance, eqs. (33)-(34)

$$\tilde\bSigma_t = \bSigma_{t-1} + \varepsilon\bI, \qquad
  \sigma_t^2 = \bx_t^{\mathsf T}\tilde\bSigma_t\bx_t, \qquad
  \bw_t = \bw_{t-1} + \frac{\tilde\bSigma_t\bx_t}{\sigma_t^2}\chi_t(e_t), \qquad
  \bSigma_t = \tilde\bSigma_t
    - \chi_t'(e_t)\frac{(\tilde\bSigma_t\bx_t)(\tilde\bSigma_t\bx_t)^{\mathsf T}}{\sigma_t^2}$$

Both share `chi_laplacian`; only $\sigma_t$ and the update direction differ.

In [ ]:
def shift(new_x_sample, x_window):
    L = len(x_window)
    new_x_window = np.zeros(L)
    new_x_window[0] = new_x_sample
    new_x_window[1:] = x_window[:-1]
    return new_x_window


def sKF_joint(n, x, d, w0, parameters):
    """Eqs. (41)-(42), scalar family. Ignacio's sKF_L_joint, rewritten only to take chi as a
    parameter; the update lines are his."""
    epsilon, b_eta = parameters["epsilon"], parameters["b_eta"]
    chi_fn = parameters.get("chi", chi_laplacian)
    L = len(w0)
    w, v = w0.copy(), parameters["var_theta_0"]
    x_t = np.zeros(L)
    w_hist, e = np.zeros((n, L)), np.zeros(n)
    for t in range(n):
        x_t = shift(x[t], x_t)
        e[t] = d[t] - x_t @ w
        w_hist[t] = w
        if t >= L:
            v_tilde = v + epsilon
            power = x_t @ x_t
            sigma = np.sqrt(v_tilde*power)             # sigma_t^2 = v~ ||x_t||^2
            chi, chi_slope = chi_fn(e[t], sigma, b_eta)
            w = w + x_t*chi/power
            v = v_tilde*(1 - chi_slope/L)
    return {"w_hist": w_hist, "e": e}


def vKF_joint(n, x, d, w0, parameters):
    """Eqs. (37)-(38), diagonal family. O(M) per step."""
    epsilon, b_eta = parameters["epsilon"], parameters["b_eta"]
    chi_fn = parameters.get("chi", chi_laplacian)
    L = len(w0)
    w = w0.copy()
    v = np.full(L, float(parameters["var_theta_0"]))   # one variance per coefficient
    x_t = np.zeros(L)
    w_hist, e = np.zeros((n, L)), np.zeros(n)
    for t in range(n):
        x_t = shift(x[t], x_t)
        e[t] = d[t] - x_t @ w
        w_hist[t] = w
        if t >= L:
            v_tilde = v + epsilon                      # v~_{t,m} = v_{t-1,m} + eps
            vx = v_tilde*x_t                           # v~_{t,m} x_{t,m}
            sigma2 = vx @ x_t                          # sigma_t^2 = sum_m v~_m x_m^2
            chi, chi_slope = chi_fn(e[t], np.sqrt(sigma2), b_eta)
            w = w + vx*(chi/sigma2)
            v = v_tilde*(1.0 - chi_slope*vx*x_t/sigma2)
            np.maximum(v, 0.0, out=v)                  # guard: chi' <= 1 keeps this non-negative
    return {"w_hist": w_hist, "e": e}


def KF_joint(n, x, d, w0, parameters):
    """Eqs. (33)-(34), full covariance. O(M^2) per step."""
    epsilon, b_eta = parameters["epsilon"], parameters["b_eta"]
    chi_fn = parameters.get("chi", chi_laplacian)
    L = len(w0)
    w = w0.copy()
    Sigma = np.eye(L)*float(parameters["var_theta_0"])
    x_t = np.zeros(L)
    w_hist, e = np.zeros((n, L)), np.zeros(n)
    eps_I = epsilon*np.eye(L)
    for t in range(n):
        x_t = shift(x[t], x_t)
        e[t] = d[t] - x_t @ w
        w_hist[t] = w
        if t >= L:
            Sigma_tilde = Sigma + eps_I                # Sigma~_t = Sigma_{t-1} + eps I
            Sx = Sigma_tilde @ x_t                     # the update direction, decorrelated
            sigma2 = x_t @ Sx                          # sigma_t^2 = x_t' Sigma~_t x_t
            chi, chi_slope = chi_fn(e[t], np.sqrt(sigma2), b_eta)
            w = w + Sx*(chi/sigma2)
            Sigma = Sigma_tilde - chi_slope*np.outer(Sx, Sx)/sigma2
            Sigma = 0.5*(Sigma + Sigma.T)              # guard: symmetry against roundoff
    return {"w_hist": w_hist, "e": e}


SCALAR = {"sKF": sKF_joint, "vKF": vKF_joint, "KF": KF_joint}
print("scalar filters ready")

In [ ]:
def _roll_in(X_t, x_win):
    x_win = np.roll(x_win, 1, axis=1)
    x_win[:, 0] = X_t
    return x_win


def sKF_batch(X, D, h, b_eta, epsilon, chi_fn=chi_laplacian):
    L, (B, n) = len(h), X.shape
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.full(B, float(VAR_THETA_0))
    eps = np.asarray(epsilon, dtype=float)
    mis, k_hist = np.empty((B, n)), np.zeros((B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = w - h
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if t < L:
            continue
        v_tilde = v + eps
        power = np.einsum("bm,bm->b", x_t, x_t)
        sigma = np.sqrt(v_tilde*power)
        k_hist[:, t] = sigma/b_eta
        chi, slope = chi_fn(e, sigma, b_eta)
        w = w + x_t*(chi/power)[:, None]
        v = v_tilde*(1 - slope/L)
    return mis, k_hist


def vKF_batch(X, D, h, b_eta, epsilon, chi_fn=chi_laplacian):
    L, (B, n) = len(h), X.shape
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    v = np.full((B, L), float(VAR_THETA_0))
    eps = np.asarray(epsilon, dtype=float)[:, None]
    mis, k_hist = np.empty((B, n)), np.zeros((B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = w - h
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if t < L:
            continue
        v_tilde = v + eps
        vx = v_tilde*x_t
        sigma2 = np.einsum("bm,bm->b", vx, x_t)
        sigma = np.sqrt(sigma2)
        k_hist[:, t] = sigma/b_eta
        chi, slope = chi_fn(e, sigma, b_eta)
        w = w + vx*(chi/sigma2)[:, None]
        v = v_tilde*(1.0 - (slope/sigma2)[:, None]*vx*x_t)
        np.maximum(v, 0.0, out=v)
    return mis, k_hist


def KF_batch(X, D, h, b_eta, epsilon, chi_fn=chi_laplacian):
    L, (B, n) = len(h), X.shape
    w, x_t = np.zeros((B, L)), np.zeros((B, L))
    Sigma = np.broadcast_to(np.eye(L)*float(VAR_THETA_0), (B, L, L)).copy()
    eps_I = np.asarray(epsilon, dtype=float)[:, None, None]*np.eye(L)
    mis, k_hist = np.empty((B, n)), np.zeros((B, n))
    for t in range(n):
        x_t = _roll_in(X[:, t], x_t)
        e = D[:, t] - np.einsum("bm,bm->b", x_t, w)
        dw = w - h
        mis[:, t] = np.einsum("bm,bm->b", dw, dw)
        if t < L:
            continue
        St = Sigma + eps_I
        Sx = np.einsum("bij,bj->bi", St, x_t)
        sigma2 = np.einsum("bi,bi->b", x_t, Sx)
        sigma = np.sqrt(sigma2)
        k_hist[:, t] = sigma/b_eta
        chi, slope = chi_fn(e, sigma, b_eta)
        w = w + Sx*(chi/sigma2)[:, None]
        Sigma = St - (slope/sigma2)[:, None, None]*np.einsum("bi,bj->bij", Sx, Sx)
        Sigma = 0.5*(Sigma + np.swapaxes(Sigma, 1, 2))
    return mis, k_hist


BATCH = {"sKF": sKF_batch, "vKF": vKF_batch, "KF": KF_batch}
print("batched filters ready")

### Checks

1. With the **Gaussian** $\chi$ of eq. (44), the KF must be the textbook Kalman filter for the
   random-walk model. That is the whole of eqs. (33)-(34) tested against an implementation written
   independently of them.
2. From a scalar prior $\bSigma_0 = v_0\bI$, the first update of all three families is the same:
   $\tilde\bSigma_t\bx_t/\sigma_t^2 = \bx_t/\|\bx_t\|^2$ when $\tilde\bSigma_t$ is a multiple of
   $\bI$. They must separate only afterwards.
3. $\chi_t' \in [0,1]$ for the Laplacian, so $\bSigma_t$ stays positive semidefinite and every
   $v_{t,m}$ stays non-negative.
4. Batched and scalar forms must agree to roundoff.

In [ ]:
def kalman_reference(n, x, d, w0, parameters):
    """Textbook Kalman filter for w_t = w_{t-1} + noise, written without reference to eqs. (33)-(34)."""
    epsilon, v_eta = parameters["epsilon"], parameters["v_eta"]
    L = len(w0)
    w, P = w0.copy(), np.eye(L)*float(parameters["var_theta_0"])
    x_t = np.zeros(L)
    w_hist = np.zeros((n, L))
    for t in range(n):
        x_t = shift(x[t], x_t)
        e = d[t] - x_t @ w
        w_hist[t] = w
        if t >= L:
            P_pred = P + epsilon*np.eye(L)
            S = x_t @ P_pred @ x_t + v_eta
            K = P_pred @ x_t/S
            w = w + K*e
            P = P_pred - np.outer(K, x_t @ P_pred)
    return {"w_hist": w_hist}


_, VAR_ETA, B_ETA = scenario(AR_A)
x_c, d_c = generate_signals(AR_A, 0, 4000)
w0 = np.zeros(M)
p_L = {"epsilon": 1e-5, "b_eta": B_ETA, "var_theta_0": VAR_THETA_0}

print("1. KF with the Gaussian chi against the textbook Kalman filter")
p_G = {"epsilon": 1e-5, "b_eta": VAR_ETA, "var_theta_0": VAR_THETA_0, "chi": chi_gaussian}
w_kf = KF_joint(4000, x_c, d_c, w0, p_G)["w_hist"]
w_ref = kalman_reference(4000, x_c, d_c, w0,
                         {"epsilon": 1e-5, "v_eta": VAR_ETA, "var_theta_0": VAR_THETA_0})["w_hist"]
print(f"   max |w_KF - w_kalman| = {np.abs(w_kf - w_ref).max():.3e}")

print("\n2. first update from a scalar prior: the three families must coincide")
first = {name: SCALAR[name](M + 1, x_c, d_c, w0, p_L)["w_hist"][-1] for name in SCALAR}
print(f"   max |vKF - sKF| = {np.abs(first['vKF'] - first['sKF']).max():.3e}")
print(f"   max |KF  - sKF| = {np.abs(first['KF'] - first['sKF']).max():.3e}")
later = {name: SCALAR[name](2000, x_c, d_c, w0, p_L)["w_hist"][-1] for name in SCALAR}
print(f"   after 2000 steps they differ: |vKF - sKF| = "
      f"{np.abs(later['vKF'] - later['sKF']).max():.3e}, "
      f"|KF - sKF| = {np.abs(later['KF'] - later['sKF']).max():.3e}")

In [ ]:
print("3. chi' stays in [0, 1] for the Laplacian")
e_grid = np.concatenate([-np.logspace(3, -6, 500), np.logspace(-6, 3, 500)])
lo, hi = np.inf, -np.inf
for k in np.logspace(-3, 2, 40):
    _, slope = chi_laplacian(e_grid, k*B_ETA, B_ETA)
    lo, hi = min(lo, slope.min()), max(hi, slope.max())
print(f"   over k_t in [1e-3, 1e2] and |e| in [1e-6, 1e3]: min = {lo:.3e}, max = {hi:.6f}")

print("\n   Sigma_t over a real run (M = 128, 4000 steps, eps = 1e-5):")
w, Sigma, x_t = w0.copy(), np.eye(M)*VAR_THETA_0, np.zeros(M)
worst_eig, worst_asym = np.inf, 0.0
for t in range(4000):
    x_t = shift(x_c[t], x_t)
    e = d_c[t] - x_t @ w
    if t >= M:
        St = Sigma + 1e-5*np.eye(M)
        Sx = St @ x_t
        s2 = x_t @ Sx
        chi, slope = chi_laplacian(e, np.sqrt(s2), B_ETA)
        w = w + Sx*(chi/s2)
        Sigma = St - slope*np.outer(Sx, Sx)/s2
        worst_asym = max(worst_asym, np.abs(Sigma - Sigma.T).max())
        Sigma = 0.5*(Sigma + Sigma.T)
        if t % 50 == 0:
            worst_eig = min(worst_eig, np.linalg.eigvalsh(Sigma).min())
print(f"   smallest eigenvalue seen = {worst_eig:.3e}, "
      f"worst asymmetry before the guard = {worst_asym:.3e}")

print("\n4. batched against scalar, misalignment curves, 3 realisations of 2000 steps")
sig = [generate_signals(AR_A, s, 2000) for s in range(3)]
Xc, Dc = np.array([s[0] for s in sig]), np.array([s[1] for s in sig])
for name in ("sKF", "vKF", "KF"):
    mis_b, _ = BATCH[name](Xc, Dc, ho, B_ETA, np.full(3, 1e-5))
    worst = 0.0
    for r in range(3):
        w_s = SCALAR[name](2000, Xc[r], Dc[r], w0, p_L)["w_hist"]
        mis_s = ((w_s - ho)**2).sum(axis=1)
        worst = max(worst, np.abs(mis_b[r] - mis_s).max()/mis_s.max())
    print(f"   {name:>4}: max relative difference = {worst:.3e}")

## 5. The sweep

Notebook 03's protocol: target first, $\varepsilon$ by grid search to land on it, floor = mean of the
last quarter, converged = first step within 3 dB of the floor. Search at $R = 3$ and
$N = 24000$, checked run at $R = 20$ and $N = 96000$. Every table reports the median and the 10-90 %
range of $k_t = \sigma_t/b_\eta$ over the checked run, since $k_t$ is the only parameter of the exact
correction and the three families reach different $\sigma_t$.

In [ ]:
EPS_GRID = np.logspace(-8, -3, 11)          # as in notebook 03


def steady_state(misalignment):
    floor = 10*np.log10(misalignment[3*len(misalignment)//4:].mean())
    return floor, int(np.argmax(10*np.log10(misalignment) < floor + 3))


def pick(grid, floors):
    best = int(np.argmin(floors))
    g, f = grid[best:], floors[best:]
    if TARGET_DB > f.max():
        return g[-1], "grid too narrow"
    if TARGET_DB < f.min():
        return g[0], "target not reached"
    order = np.argsort(f)
    return 10**np.interp(TARGET_DB, f[order], np.log10(g)[order]), \
        ("ok" if best > 0 else "ok (optimum at grid edge)")


def sweep(name, X_search, D_search, X_check, D_check, b_eta):
    """Grid search then the checked run, for one filter at one input correlation."""
    fn = BATCH[name]
    B = len(EPS_GRID)
    Xs = np.repeat(X_search, B, axis=0)
    Ds = np.repeat(D_search, B, axis=0)
    mis, _ = fn(Xs, Ds, ho, b_eta, np.tile(EPS_GRID, len(X_search)))
    floors = np.array([steady_state(c)[0]
                       for c in mis.reshape(len(X_search), B, -1).mean(axis=0)])
    eps, status = pick(EPS_GRID, floors)
    mis, k_hist = fn(X_check, D_check, ho, b_eta, np.full(len(X_check), eps))
    floor, steps = steady_state(mis.mean(axis=0))
    k = k_hist[:, M:].ravel()
    return dict(epsilon=eps, floor=floor, steps=steps, status=status,
                k_med=np.median(k), k_lo=np.percentile(k, 10), k_hi=np.percentile(k, 90),
                curve=mis.mean(axis=0))


def run_all(ar_a, names=("sKF", "vKF", "KF"), convention="variance"):
    _, _, b_eta = scenario(ar_a, convention)
    search = [generate_signals(ar_a, s, N_SEARCH) for s in range(R_SEARCH)]
    check = [generate_signals(ar_a, s, N) for s in range(R)]
    Xs, Ds = np.array([s[0] for s in search]), np.array([s[1] for s in search])
    Xc, Dc = np.array([s[0] for s in check]), np.array([s[1] for s in check])
    out = {}
    for name in names:
        t0 = time.time()
        out[name] = sweep(name, Xs, Ds, Xc, Dc, b_eta)
        out[name]["seconds"] = time.time() - t0
        print(f"   {name} done in {out[name]['seconds']:.0f} s")
    return out


def table(results, title):
    print(f"\n{title}")
    print(f"{'filter':>6}{'epsilon':>11}{'floor [dB]':>12}{'steps':>8}"
          f"{'k_t median':>12}{'k_t 10-90%':>18}  status")
    for name, r in results.items():
        span = f"{r['k_lo']:.3f} - {r['k_hi']:.3f}"
        print(f"{name:>6}{r['epsilon']:>11.2e}{r['floor']:>12.2f}{r['steps']:>8d}"
              f"{r['k_med']:>12.3f}{span:>18}   {r['status']}")


print("sweep ready")

In [ ]:
start = time.time()
print(f"AR({AR_A}) input, M = {M}, SNR {SNR_DB:.0f} dB, target {TARGET_DB:.0f} dB")
res_ar = run_all(AR_A)
table(res_ar, f"AR({AR_A}) input")
print(f"\nsteps relative to the sKF:  "
      + ",  ".join(f"{n} {res_ar[n]['steps']/res_ar['sKF']['steps']:.3f}"
                   for n in ("vKF", "KF")))
print(f"\n{time.time() - start:.0f} s")

In [ ]:
start = time.time()
print(f"white input, M = {M}, SNR {SNR_DB:.0f} dB, target {TARGET_DB:.0f} dB")
res_white = run_all(0.0)
table(res_white, "white input")
print(f"\nsteps relative to the sKF:  "
      + ",  ".join(f"{n} {res_white[n]['steps']/res_white['sKF']['steps']:.3f}"
                   for n in ("vKF", "KF")))
print(f"\n{time.time() - start:.0f} s")

## 6. Does the KF recover the coloured-input loss?

Section 4 claims it does, without prewhitening. The test: how much of the sKF's coloured-input
penalty each family gives back. The penalty is the sKF's step count under AR($-0.9$) over its step
count under white input; a filter that recovers the loss in full would match the white-input sKF.

The white-input runs use the same $\bh_o$ and the same SNR, so the signal power differs only through
$\bh_o^{\mathsf T}\bR_{xx}\bh_o$; that is stated with the numbers in section 2.

In [ ]:
base = res_white["sKF"]["steps"]
print(f"steps to converge at equal floor ({TARGET_DB:.0f} dB), SNR {SNR_DB:.0f} dB\n")
print(f"{'filter':>6}{'white':>9}{'AR(-0.9)':>11}{'colour penalty':>16}"
      f"{'AR steps / white sKF':>22}")
for name in ("sKF", "vKF", "KF"):
    w_, a_ = res_white[name]["steps"], res_ar[name]["steps"]
    print(f"{name:>6}{w_:>9d}{a_:>11d}{a_/w_:>16.2f}{a_/base:>22.2f}")

penalty = res_ar["sKF"]["steps"]/res_white["sKF"]["steps"]
recovered = {n: (res_ar["sKF"]["steps"] - res_ar[n]["steps"])
                / (res_ar["sKF"]["steps"] - base) for n in ("vKF", "KF")}
print(f"\nthe sKF's colour penalty is {penalty:.2f}x")
for n in ("vKF", "KF"):
    print(f"the {n} gives back {100*recovered[n]:.0f}% of it")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)
t_axis = np.arange(N)/FS

for ax, (res, title) in zip(axes, [(res_ar, f"AR({AR_A}) input"), (res_white, "white input")]):
    for (name, r), colour in zip(res.items(), COLORS):
        ax.plot(t_axis, 10*np.log10(r["curve"]), lw=1.1, color=colour,
                label=f"{name}, {r['steps']} steps")
    ax.axhline(TARGET_DB, color="k", ls="--", lw=0.8)
    ax.set(xlabel="time [s]", ylabel=r"misalignment [dB]", title=title, ylim=(-25, 5))
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.show()

## 7. The prewhitened sKF

The fourth check of issue #11: does the KF match what prewhitening would buy? Filtering both $x_t$
and $d_t$ by $A(z) = 1 - a z^{-1}$ recovers the white driving noise $u_t$ and leaves the same
$\bh_o$ to identify, but it **colours the observation noise**, $\eta_t \to \eta_t - a\eta_{t-1}$,
whose variance grows by $1 + a^2 = 1.81$ and which is no longer i.i.d. The filters assume i.i.d.
noise, so this comparison is informative but not like-for-like; the noise variance and the scale
$b_\eta$ below are recomputed from the filtered noise, and the effective SNR is reported with them.
That mismatch is the point: the KF needs none of it.

In [ ]:
def prewhiten(ar_a, seed, n):
    """A(z) = 1 - a z^-1 applied to x and d. Returns the filtered pair and the filtered noise."""
    x, d = generate_signals(ar_a, seed, n)
    a = [1.0, -ar_a]
    return lfilter(a, [1.0], x), lfilter(a, [1.0], d)


start = time.time()
_, var_eta_ar, _ = scenario(AR_A)
var_eta_pw = var_eta_ar*(1 + AR_A**2)            # noise variance after A(z)
b_eta_pw = np.sqrt(var_eta_pw/2)
snr_pw = 10*np.log10(1.0/var_eta_pw)             # the driving noise u has unit variance

print(f"after prewhitening: var_eta {var_eta_ar:.4e} -> {var_eta_pw:.4e}, "
      f"b_eta {scenario(AR_A)[2]:.4f} -> {b_eta_pw:.4f}")
print(f"effective SNR {SNR_DB:.1f} dB -> {snr_pw:.1f} dB, and the noise is no longer i.i.d.\n")

search_pw = [prewhiten(AR_A, s, N_SEARCH) for s in range(R_SEARCH)]
check_pw = [prewhiten(AR_A, s, N) for s in range(R)]
Xs_pw, Ds_pw = np.array([s[0] for s in search_pw]), np.array([s[1] for s in search_pw])
Xc_pw, Dc_pw = np.array([s[0] for s in check_pw]), np.array([s[1] for s in check_pw])
res_pw = {"sKF": sweep("sKF", Xs_pw, Ds_pw, Xc_pw, Dc_pw, b_eta_pw)}
table(res_pw, "sKF on the prewhitened AR(-0.9) input")

print(f"\n{'route':>34}{'steps':>8}")
print(f"{'sKF, coloured input':>34}{res_ar['sKF']['steps']:>8d}")
print(f"{'sKF, prewhitened input':>34}{res_pw['sKF']['steps']:>8d}")
print(f"{'KF, coloured input, no prewhitening':>34}{res_ar['KF']['steps']:>8d}")
print(f"{'sKF, white input (the reference)':>34}{res_white['sKF']['steps']:>8d}")
print(f"\n{time.time() - start:.0f} s")

## 8. Issue #9: $b_\eta = \mathrm{E}|\eta_t|$

Every run above tells the filter $b_\eta = \sqrt{v_\eta/2}$, matching the noise **variance**, as
every notebook up to 07 does. The new draft uses $b_\eta = \mathrm{E}|\eta_t|$, matching the **mean
absolute deviation**, which is the maximum-likelihood and KL-closest Laplacian fit to the noise we
actually simulate. That is the convention to adopt: Section 2.3 already projects onto a family by
minimizing KL, and applying the same rule to the likelihood model gives the Gaussian filter $v_\eta$
and the Laplacian filter $\mathrm{E}|\eta_t|$, so each is handed the closest member of its own family.

$k_t = \sigma_t/b_\eta$ is the only parameter of the exact correction, so the change moves every
result along the $k$ family. At $\beta^* = 1$ the two coincide exactly, $\mathrm{E}|\eta| =
\sqrt{v_\eta/2}$ being an identity for the Laplacian, so nothing at $\beta^* = 1$ needs re-running.

In [ ]:
print(f"{'beta*':>7}{'E|eta|/sqrt(v)':>17}{'b_mad/b_var':>14}")
for beta in (0.2, 1.0, 2.0):
    ratio = (np.exp(gammaln(2/beta) - gammaln(1/beta))
             / np.sqrt(np.exp(gammaln(3/beta) - gammaln(1/beta))))
    print(f"{beta:>7.1f}{ratio:>17.4f}{ratio/np.sqrt(0.5):>14.4f}")

_, v_ar, b_var = scenario(AR_A, "variance")
_, _, b_mad = scenario(AR_A, "mad")
print(f"\nat this operating point (AR({AR_A}), SNR {SNR_DB:.0f} dB, beta* = {BETA}):")
print(f"   var_eta               = {v_ar:.5f}")
print(f"   b_eta, variance-matched = {b_var:.5f}")
print(f"   b_eta, E|eta|           = {b_mad:.5f}")
print(f"   every k_t grows by      {b_var/b_mad:.3f}x")

In [ ]:
start = time.time()
print(f"AR({AR_A}) input at b_eta = E|eta|")
res_mad = run_all(AR_A, convention="mad")
table(res_mad, f"AR({AR_A}) input, b_eta = E|eta|")

print(f"\nboth conventions side by side, AR({AR_A}), SNR {SNR_DB:.0f} dB, target {TARGET_DB:.0f} dB\n")
print(f"{'filter':>6}{'steps, b=sqrt(v/2)':>21}{'steps, b=E|eta|':>18}{'change':>9}"
      f"{'k_t med, var':>14}{'k_t med, mad':>14}")
for name in ("sKF", "vKF", "KF"):
    a, b = res_ar[name], res_mad[name]
    print(f"{name:>6}{a['steps']:>21d}{b['steps']:>18d}{b['steps']/a['steps']:>9.2f}"
          f"{a['k_med']:>14.3f}{b['k_med']:>14.3f}")
print(f"\n{time.time() - start:.0f} s")

## 9. Findings

**vKF, issue #10.** The diagonal family lands on the scalar one: 5659 steps against the sKF's 5787
under AR($-0.9$), 1508 against 1511 under white input, and the same $k_t$ to three decimals. Two
percent, inside the protocol's own noise. One variance per coefficient buys nothing in this
scenario, because the regressor is a sliding window of one stationary process, so every tap sees the
same marginal statistics and the $v_{t,m}$ stay nearly equal. It is where the taps differ
systematically that the diagonal family should pay: a sparse or block-structured response, or an
input whose spectrum is far from flat across the window.

**KF, issue #11.** It recovers the coloured-input loss in full, without prewhitening, as Section 4
claims.

| route | steps | |
|---|---|---|
| sKF, white input | 1511 | the reference: no colour to lose |
| sKF, AR($-0.9$) | 5787 | a colour penalty of 3.83x |
| sKF, prewhitened AR($-0.9$) | 2621 | and 1.7 dB of SNR, and the i.i.d. noise |
| **KF, AR($-0.9$), no prewhitening** | **1521** | within 1% of the white-input sKF |

The KF gives back 99.8% of the penalty. Explicit prewhitening gives back 74%, while costing
1.7 dB of effective SNR and the i.i.d. assumption the filters are built on, since
$\eta_t \to \eta_t - a\eta_{t-1}$. The mechanism is the one Section 4 names: the update direction
$\tilde\bSigma_t\bx_t$ is decorrelated, and $\sigma_t^2 = \bx_t^{\mathsf T}\tilde\bSigma_t\bx_t$ is
then the right predicted variance in every direction rather than in the average one.

The KF is also 2.3x faster than the sKF on **white** input, 651 steps against 1511, so the full
covariance is worth having even where there is no colour to remove. Its $k_t$ sits lower and much
tighter than the sKF's (median 0.508, 10-90% 0.460-0.559, against 0.938 and 0.750-1.180): it reaches
the same floor from a smaller and steadier predicted spread.

**Correctness.** With the Gaussian $\chi$ of eq. (44) the KF reproduces a textbook Kalman filter,
written independently of eqs. (33)-(34), to $1.3\times10^{-14}$. All three families agree exactly on
the first update from a scalar prior and separate only afterwards. $\chi_t'$ stays inside $[0,1]$
over $k_t \in [10^{-3}, 10^2]$ and $|e| \in [10^{-6}, 10^3]$, so $\bSigma_t$ stays positive
semidefinite; the smallest eigenvalue seen over a real run is $4.8\times10^{-5}$, and the rank-one
downdate is exactly symmetric before the guard. Batched and scalar forms agree to $10^{-10}$
relative.

**Convention, issue #9.** Switching to $b_\eta = \mathrm{E}|\eta_t|$ multiplies every $k_t$ by 2.82
and makes every filter about 18% faster to the same floor. The change is uniform across the three
families (0.82, 0.83, 0.82), so nothing in the comparison between them moves: the KF's recovery of
the colour penalty holds under either convention. $\beta^* = 1$ needs no re-run, $\mathrm{E}|\eta| =
\sqrt{v_\eta/2}$ being an identity for the Laplacian.

**Cost.** At $M = 128$ batched over $R = 20$, the KF runs at 36 us per step per realisation against
1.6 for the sKF and 1.9 for the vKF, so one sweep is about 110 s against 5 s. That is the price of
the 3.8x.

**One caveat on the sweeps.** Both KF rows, and the white-input sKF and vKF, report "optimum at grid
edge": the floor is still falling at the smallest $\varepsilon$ on notebook 03's grid,
$10^{-8}$. The selected $\varepsilon$ is interior in every case and the target is bracketed, so the
interpolation is sound, but a deeper target than $-20$ dB would need the grid extended downward.

In [ ]:
print(f"total notebook time: {time.time() - NOTEBOOK_START:.0f} s")